# PDF RAG with ChromaDB + Local Embeddings

This notebook loads a PDF, chunks it, embeds it using a local LM Studio model,
and stores everything in a Chroma vector database for similarity search.

## 1. Install Dependencies

In [34]:
!pip install langchain-chroma langchain-openai python-dotenv pypdf

## 2. Configuration

Set your PDF path and chunking parameters below.

In [35]:
PDF_PATH = "your_file.pdf"   # <-- change this to your PDF
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
EMBEDDING_MODEL = "text-embedding-nomic-embed-text-v1.5@q4_k_m"  # model loaded in LM Studio
LM_STUDIO_URL = "http://localhost:1234/v1"

## 3. Imports

In [36]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

## 4. Load PDF

In [38]:
loader = PyPDFLoader("/Users/adithya/adit/h2/Geographical Indications of Goods Act 1999.pdf")
docs = loader.load()
print(f"Loaded {len(docs)} pages from PDF")

Loaded 31 pages from PDF


## 5. Split into Chunks

In [39]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

Split into 252 chunks


## 6. Set Up Embeddings + Vector Store

Make sure LM Studio is running with the embedding model loaded.
Go to Developer tab -> load your embedding model -> start the server.

In [40]:
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key="lm-studio",
    base_url=LM_STUDIO_URL,
    check_embedding_ctx_length=False,
    model_kwargs={"encoding_format": "float"}
)

vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="my_chroma_db",
    collection_name="pdf_collection"
)
print("Vector store ready")

Vector store ready


## 7. Add Chunks to Vector Store

In [41]:
vector_store.add_documents(chunks)
print(f"Added {len(chunks)} chunks to vector store")

Added 252 chunks to vector store


## 8. Similarity Search

Ask a question and get back the most relevant chunks from your PDF.

In [42]:
query = "Meaning of applying geographical indications"  # <-- change this
results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results, 1):
    print(f"--- Result {i} (page {doc.metadata.get('page', '?')}) ---")
    print(doc.page_content[:300])
    print()

--- Result 1 (page 16) ---
CHAPTER VIII 
OFFENCES, PENALTIES AND PROCEDURE 
37. Meaning of applying geographical indications .—(1) A person shall be deemed to apply a 
geographical indication to goods who— 
(a) applies it to the goods themselves; or 
(b) applies it to any package in or with which the goods are sold, or expose

--- Result 2 (page 8) ---
(a) a statement as to how the geographical indication serves to designate the goods as originating 
from the concerned territory of the country or region or locality in the country, as the case may be, in 
respect of specific quality, reputation or other characteristics of which are due exclusively 

--- Result 3 (page 6) ---
the use of the geographical indication upon, or in any physical or in any other rela tion whatsoever, to 
such goods; 
(c) to a registered geographical indication shall be construed as including a reference to a 
geographical indication registered in the register;  
(d) to the Registrar shall be con



## 9. Search with Relevance Scores

Lower score = more similar.

In [43]:
results_with_scores = vector_store.similarity_search_with_score(query, k=3)

for doc, score in results_with_scores:
    print(f"Score: {score:.4f} | {doc.page_content[:150]}...")
    print()

Score: 0.2827 | CHAPTER VIII 
OFFENCES, PENALTIES AND PROCEDURE 
37. Meaning of applying geographical indications .—(1) A person shall be deemed to apply a 
geographi...

Score: 0.3661 | (a) a statement as to how the geographical indication serves to designate the goods as originating 
from the concerned territory of the country or reg...

Score: 0.4010 | the use of the geographical indication upon, or in any physical or in any other rela tion whatsoever, to 
such goods; 
(c) to a registered geographica...

